# Setting up a Continuation Power Flow Calculation

## Imports and Definitions

In [ ]:
# %matplotlib widget
# import ipympl

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler/")
sys.path.insert(1, parent)
sys.path.append(
    str(
        os.path.dirname(
            os.path.dirname(os.path.abspath("continuation_power_flow.ipynb"))
        )
    )
)
save = os.path.join(os.getcwd(), "plots/")

import matplotlib as mpl
from tools import *

In [ ]:
import development_files.examples.ibb_transformer.ibb_trans_model as mdl
from src.diffpssi.power_sim_lib.simulator import PowerSystemSimulation as Pss
from src.diffpssi.power_sim_lib.simulator import Recorder
from src.diffpssi.power_sim_lib.load_flow import do_load_flow
from src.diffpssi.stability_lib.voltage import NoseCurve

## Define Test System

In [ ]:
def test_system(trans_type="simple", control="oltc", B1=[400, 0], param_dict_oltc=None):
    return {
        "base_mva": 2200,
        "f": 60,
        "slack_bus": "B0",
        "base_voltage": 100,
        "busses": [
            ["name", "V_n"],
            ["B0", 10],
            ["B1", 100],
        ],
        "transformers": [
            [
                "type",
                "control",
                "name",
                "from_bus",
                "to_bus",
                "S_n",
                "tap_side",
                "measure_side",
                "V_n_from",
                "V_n_to",
                "R",
                "X",
                "param_dict_oltc",
            ],
            [
                trans_type,
                control,
                "T1",
                "B0",
                "B1",
                2200,
                "hv",
                "hv",
                10,
                100,
                0,
                0.15,
                param_dict_oltc,
            ],
        ],
        "generators": {
            "GEN": [
                [
                    "name",
                    "bus",
                    "S_n",
                    "V_n",
                    "P",
                    "V",
                    "H",
                    "D",
                    "X_d",
                    "X_q",
                    "X_d_t",
                    "X_q_t",
                    "X_d_st",
                    "X_q_st",
                    "T_d0_t",
                    "T_q0_t",
                    "T_d0_st",
                    "T_q0_st",
                ],
                [
                    "G1",
                    "B0",
                    2200,
                    10,
                    -1998,
                    1,
                    3.5,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
            ],
        },
        "loads": {
            "ZIP": [
                ["name", "bus", "P", "Q", "model"],
                ["L1", "B1", B1[0], B1[1], "Z"],
            ],
        },
    }

## Set Up Sim

In [ ]:
param_dict_oltc = {
    "t_1": 5,
    "db": 0.05,
    "delta_m": 0.02,
    "m_max": 1.1,
    "m_min": 0.9,
    "v_ref": 1,
}

parallel_sims = 1
sim = Pss(
    parallel_sims=parallel_sims,
    sim_time=300,
    time_step=0.005,
    verbose=False,
)

## Simple Grid and its Nose Curve Calculation

In [ ]:
p_load = np.linspace(0, 10000, 1000)
# tan_phi = np.linspace(-0.2, 5, 8)
tan_phi = [-0.4, -0.2, 0, 0.2, 1, 1.5]

nose_curve = NoseCurve(
    load_model=test_system,
    loading={"p": p_load, "tan_phi": tan_phi},
)

result_mesh = nose_curve.run_calculation(bus=["B1"])["B1"]

ax = nose_curve.plot_nose_curve(
    ["B1"], size=(8, 5), title=False, save_path=save + "simple_load_"
)
plt.show()

## Loadability of a More Complex System - IEEE 9-bus

In [ ]:
from development_files.examples.ieee_9bus.ieee_9bus_model import load

p_load = np.linspace(0, 750, 1000)
tan_phi = [-0.2, 0, 0.2, 0.4, 1]
# tan_phi = [0]

nose_curve_9bus = NoseCurve(
    load_model=load,
    loading={"p": p_load, "tan_phi": tan_phi},
)

result_mesh = nose_curve_9bus.run_calculation(["B5", "B6", "B8"])["B5"]  # , 'B6', 'B8'

ax = nose_curve_9bus.plot_nose_curve(
    ["B5", "B6", "B8"], size=(8, 5), title=False, save_path=save + "9bus_load_"
)
plt.show()

In [ ]:
nose_curve_9bus.result

max_load = nose_curve_9bus.get_max_loadings(busses=["B5", "B6", "B8"])
print(max_load["B5"][0])

In [ ]:
# nose_curve_9bus.reset_sim_parameters()
# nose_curve_9bus.ps_sim.create_grid(load())
# do_load_flow(nose_curve_9bus.ps_sim)

# bus_voltages = [bus.voltage.squeeze() for bus in nose_curve_9bus.ps_sim.busses]

# print('Initial Bus Voltages for the base load case:')
# for i, v in enumerate(bus_voltages):
#     print(f"{f'Voltage @ Bus {i}: ':<20}{np.round(np.abs(v), 2)}")

## Comparison to PowerFactory Results

Import Data from ```.csv``` file: Result from PowerFactory.

In [ ]:
pf_nc_bus5 = pd.read_csv("./data/nose_curve_b5_load_scale.csv", sep=";", dtype=float)
pf_nc_bus5.rename(
    columns={"b:totDemand": "p", "m:u": "v", "b:uGradient": "v-p"}, inplace=True
)
# pf_nc_bus5.set_index('t', inplace=True)

pf_nc_bus5["p"] = pf_nc_bus5["p"] / 1e6

pf_nc_bus5.head()

In [ ]:
from development_files.examples.ieee_9bus.ieee_9bus_model import load

p_load = np.linspace(0, 750, 1000)
tan_phi = [0.4]

nose_curve_9bus_valid = NoseCurve(
    load_model=load,
    loading={"p": p_load, "tan_phi": tan_phi},
)

result_mesh = nose_curve_9bus_valid.run_calculation(["B5"])["B5"]
print(result_mesh)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    result_mesh["p"],
    np.abs(result_mesh["v"]),
    label=r"diffpssi Result for $\tan \phi = 0.4$",
)
plt.plot(
    pf_nc_bus5["p"], pf_nc_bus5["v"], label=r"PowerFactory Result for $\tan \phi = 0.4$"
)

plt.grid()
plt.legend()
plt.ylabel("Voltage Magnitude in p.u.")
plt.xlabel("Power in MW")

plt.savefig("./plots/9bus_comp_b5.pdf")

plt.show()